## 6 — MRdeeP (state-level estimates, CES sample2 / 3000)
Multivariate Multilevel Regression with Deep Generative Post-Stratification.
Outcomes extracted: `climate_problem`, `renewable_fuel`.

Pipeline:
1. `insert_data` — encodes CES survey + county-level benchmark
2. `fit` — trains an ensemble of CGANs (Wasserstein loss + gradient penalty)
3. `post_stratify('state_fips')` — generates synthetic micro-data per demographic
   cell, groups by state → extracts `climate_problem` and `renewable_fuel` estimates

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from pathlib import Path

os.environ['DEEPVERSE_BACKEND'] = 'pytorch'
sys.path.insert(0, '/Users/carmenk/Documents/CSS/Capstone/mrdeep/python')
from mrdeep import MRdeeP

sys.path.insert(0, str(Path('.').resolve()))
from utils import OUTPUT_DIR, STATE_FIPS_TO_NAME, SURVEY_PATH, save_estimates

DATA_DIR   = Path("../../")
OUTCOME    = ['climate_problem', 'renewable_fuel']
MODEL_NAME = 'mrdeep'

### 1. Load and prepare data

In [2]:
OUTCOME_COLS = ['climate_problem','regulate_carbon','renewable_fuel',
                'clean_air_water','fuel_efficiency','fossil_fuel','paris_agreement']
DEMOG_VARS   = ['gender', 'race4', 'educ_category', 'county_fips', 'state_fips']

raw       = pd.read_csv(SURVEY_PATH, dtype={'state_fips': str, 'county_fips': str})
ps_county = pd.read_csv(DATA_DIR / 'post_stratification_frame' / 'poststrat_county.csv',
                        dtype={'state_fips': str, 'county_fips': str})

raw['county_fips'] = raw['county_fips'].astype(str).str.zfill(5)
OUTCOME_COLS = [c for c in OUTCOME_COLS if c in raw.columns]

survey = raw[DEMOG_VARS + OUTCOME_COLS].dropna().copy()
survey['educ_category'] = survey['educ_category'].astype(str)

benchmark = ps_county[DEMOG_VARS + ['N_rounded']].copy()
benchmark['educ_category'] = benchmark['educ_category'].astype(str)
target_rows = len(benchmark)
benchmark['count'] = np.maximum(
    1,
    (benchmark['N_rounded'] / benchmark['N_rounded'].sum() * target_rows).round(),
).astype(int)
benchmark = benchmark.drop(columns=['N_rounded'])

print(f'Survey (complete cases): {len(survey):,}')
for oc in OUTCOME_COLS:
    print(f'  {oc}: {survey[oc].mean()*100:.1f}% support')
print(f'Benchmark strata: {len(benchmark):,}  augmented rows: {benchmark["count"].sum():,}')

Survey (complete cases): 2,977
  climate_problem: 62.8% support
  regulate_carbon: 65.1% support
  renewable_fuel: 60.0% support
  clean_air_water: 56.5% support
  fuel_efficiency: 65.9% support
  fossil_fuel: 62.9% support
  paris_agreement: 58.6% support
Benchmark strata: 99,940  augmented rows: 170,690


### 2. Insert data into MRdeeP

In [3]:
mod = MRdeeP(ensembles=3, random_state=42)

mod.insert_data(
    survey     = survey,
    benchmark  = benchmark,
    demog_vars = DEMOG_VARS,
    count_col  = 'count',
    oversample = 1,
)
print(mod)

MRdeeP (backend=pytorch, ensembles=3)
  Data inserted: True
  Survey: 2977 obs, 7 substantive vars, 5 demographic vars
  Augmented benchmark: 170690 rows
  Fitted: False


### 3. Train CGAN ensemble
Default architecture: 4 × 256-neuron hidden layers, Wasserstein loss + gradient penalty.

In [4]:
mod.fit(
    epochs        = 500,
    patience      = 50,
    batch_size    = 256,
    k             = 32,
    print_runtime = True,
)
print(mod)

Ensemble 1/3


Ensemble 2/3


Ensemble 3/3


Total fit time: 410.1 seconds.
MRdeeP (backend=pytorch, ensembles=3)
  Data inserted: True
  Survey: 2977 obs, 7 substantive vars, 5 demographic vars
  Augmented benchmark: 170690 rows
  Fitted: True
    Noise dim (k): 32
    Ensemble members: 3
    Generated survey: 170690 rows
    Total fit time: 410.1s


### 4. Post-stratify → state-level estimates for all outcomes

In [5]:
estimates = mod.post_stratify(levels='state_fips')
print(f'Estimates shape: {estimates.shape}  ({estimates["state_fips"].nunique()} states)')
estimates.head()

Estimates shape: (51, 8)  (51 states)


,state_fips,clean_air_water,climate_problem,fossil_fuel,fuel_efficiency,paris_agreement,regulate_carbon,renewable_fuel
0,01,0.496902,0.571085,0.639800,0.660030,0.616404,0.630723,0.504317
1,02,0.505687,0.574215,0.636840,0.661672,0.623970,0.630745,0.510054
2,04,0.509683,0.580256,0.629110,0.663666,0.625907,0.635448,0.515877
3,05,0.496359,0.569387,0.639794,0.658933,0.616097,0.630223,0.502011
4,06,0.512479,0.581663,0.635849,0.671403,0.627679,0.641264,0.516278


### 5. Extract target outcomes and save

In [6]:
for OUTCOME_VAR in OUTCOME:
    result = estimates[['state_fips', OUTCOME_VAR]].rename(
        columns={OUTCOME_VAR: 'estimate'}
    ).copy()
    result['state_name'] = result['state_fips'].map(STATE_FIPS_TO_NAME)

    save_estimates(result, MODEL_NAME, OUTCOME_VAR)

    print(f'\n--- {OUTCOME_VAR} ---')
    print(f'National mean: {result["estimate"].mean():.3f}')
    print(result.sort_values("estimate", ascending=False).head(5)[["state_name","estimate"]].to_string(index=False))


  mrdeep (climate_problem) — State-Level Estimates
  States with estimates: 51
  States with NaN:       0
  Mean estimate:         0.5710
  Median estimate:       0.5695
  Min estimate:          0.5451
  Max estimate:          0.6087

  Saved → /Users/kaeleyoshea/Capstone/A.MRdeeP-Deep-Learning--MRP/model_run_ces/sample2_state/outputs/estimates/climate_problem_state_estimates.csv


--- climate_problem ---
National mean: 0.571
   state_name  estimate
Massachusetts  0.608745
 Rhode Island  0.595514
     Illinois  0.585786
   Washington  0.584076
     Oklahoma  0.581896

  mrdeep (renewable_fuel) — State-Level Estimates
  States with estimates: 51
  States with NaN:       0
  Mean estimate:         0.5055
  Median estimate:       0.5040
  Min estimate:          0.4888
  Max estimate:          0.5394

  Saved → /Users/kaeleyoshea/Capstone/A.MRdeeP-Deep-Learning--MRP/model_run_ces/sample2_state/outputs/estimates/renewable_fuel_state_estimates.csv


--- renewable_fuel ---
National mean: 0.5